# LLMs Alignment with RLHF


Here, we're gonna fine-tune a LLM with reinforcement learning to make it generate negative movie reviews

For fine-tuning with RL we'll use [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl) library. It implements the main RLHF components: reward modeling and PPO fine-tuning


We'll use GPT2 for aligning it on IMDB datasaet to generate bad reviews. But before the process, let's have a look at the baseline model, which is fine-tuned on generating arbitrary movie reviews.

In [ ]:
import transformers
import torch

runtime_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
policy_model = transformers.AutoModelForCausalLM.from_pretrained(
    "lvwerra/gpt2-imdb", device_map=runtime_device
)

tokenizer_config.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
baseline_input = policy_tokenizer("The movie", return_tensors="pt").to(runtime_device)
baseline_tokens = policy_model.generate(
    **baseline_input, do_sample=True, max_new_tokens=50
)
decoded_baseline = policy_tokenizer.decode(baseline_tokens.flatten().cpu().numpy().tolist())
print("\nGenerated text:", decoded_baseline)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



Generated text: The movie is a huge failure of a first rate script and production. It should have been rated at least 12,000 and had to be removed from the same level (if I remember correctly). I have never witnessed one movie as good in my life. No


If you run this cell several times, you can see that the model generates positive, negative and neutral reviews in natural proportion. We'll change that, by teaching the model to generate more negative reviews.

We will accomplish this in two stages:
- train a reward model to assign higher values to negative reviews
- fine-tune the LLM to maximize the reward using PPO algorithm

## 1. Training a reward model

We'll fine-tune a BERT-like model as a reward model. For this, we need to generate a synthetic pair rankings to imitate human rankings.


In [ ]:
scorer_model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-cased", device_map=runtime_device
)
scorer_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
# A dataset of positive-negative pairs.

import torch
import datasets

class ReviewPairsDataset(torch.utils.data.Dataset):
    """All chosen/rejected review combinations in TRL reward-training format."""

    def __init__(self, reviews, tokenizer, preferred_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.preferred_reviews = [item["text"] for item in reviews if item["label"] == preferred_label]
        self.other_reviews = [item["text"] for item in reviews if item["label"] != preferred_label]
        assert self.preferred_reviews, f"no texts with label {preferred_label}"
        print(
            f"Found {len(self.preferred_reviews)} chosen and {len(self.other_reviews)} rejected texts, "
            f"{len(self)} pairs"
        )

    def __len__(self):
        return len(self.preferred_reviews) * len(self.other_reviews)  # all pairs

    def __getitem__(self, index: int):
        preferred = self.tokenizer(
            self.preferred_reviews[index // len(self.preferred_reviews)], truncation=True
        )
        other = self.tokenizer(
            self.other_reviews[index % len(self.preferred_reviews)], truncation=True
        )
        return {
            "input_ids_chosen": preferred["input_ids"],
            "attention_mask_chosen": preferred["attention_mask"],
            "input_ids_rejected": other["input_ids"],
            "attention_mask_rejected": other["attention_mask"],
        }

In [ ]:
PREFERRED_LABEL = 0
train_reviews = datasets.load_dataset("imdb", split="train")
preference_pairs = ReviewPairsDataset(
    train_reviews, scorer_tokenizer, preferred_label=PREFERRED_LABEL
)

example_pair = preference_pairs[31337]
print("CHOSEN:", scorer_tokenizer.decode(example_pair["input_ids_chosen"]))
print("REJECTED:", scorer_tokenizer.decode(example_pair["input_ids_rejected"]))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Found 12500 chosen and 12500 rejected texts, 156250000 pairs
CHOSEN: [CLS] If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story. < br / > < br / > One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives ( unless one comes up with one while one ' s mind wanders, as it will invariably do during this pointless film ). < br / > < br / > One might better spend one ' s time staring out a window at a tree growing. < br / > < br / > [SEP]
REJECTED: [CLS] This movie has some things that are pretty amazing. First, it is supposed to be based on a true story. That, in itself, is amazing that multiple tornadoes would hit the same town at night in the fall - in Nebraska. I wonder if the real town ' s name was close to " Blainsworth " ( which is the town ' s name in the movie ). There is an Ainsworth, N

We'll use `trl.RewardTrainer`, which uses the following objective from [the InstructGPT paper](https://arxiv.org/pdf/2203.02155.pdf):
![img](https://i.imgur.com/2JzNAPs.png)

The model itself does not score pairs: the chosen ($y_w$) and the rejected ($y_l$) samples are processed independently. The reward model needs to score the chosen sample higher than the rejected one.

In [ ]:
import trl

reward_config = trl.RewardConfig(
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,
    logging_steps=50,
    gradient_checkpointing=True,  # reducing memory usage
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True
)

reward_trainer = trl.RewardTrainer(
    model=scorer_model,
    args=reward_config,
    tokenizer=scorer_tokenizer,
    train_dataset=preference_pairs,
    peft_config=None,
)

reward_trainer.train()

/usr/local/lib/python3.11/dist-packages/trl/trainer/reward_trainer.py:175: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/reward_trainer.py:192: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` stat

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
50,0.535600
100,0.207400
150,0.150400
200,0.134900
250,0.101900
300,0.106400
350,0.097100
400,0.088800
450,0.085600
500,0.081400


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=0.11340342354774476, metrics={'train_runtime': 1536.7251, 'train_samples_per_second': 20.824, 'train_steps_per_second': 0.651, 'total_flos': 0.0, 'train_loss': 0.11340342354774476, 'epoch': 0.00020479997902848215})

In [ ]:
scorer_model.gradient_checkpointing_disable()
scorer_model.eval()

### Sanity check of the reward model

To check how our reward model performs, we'll measure how often it ranks a pair of chosen and rejected reviews correctly. We'll do this separately for the train and test sets.


In [ ]:
for review_index in (45, 16000):
    review = train_reviews[review_index]
    print("TEXT:", review["text"])
    encoded_review = scorer_tokenizer(
        review["text"], truncation=True, return_tensors="pt"
    ).to(runtime_device)
    with torch.no_grad():
        review_reward = scorer_model(**encoded_review).logits[0, 0].item()
        print("REWARD:", review_reward)
    print("LABEL:", review["label"])
    print()

TEXT: This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get "terrorized" by this pathetic "crazed killer", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.
REWARD: 4.8359375
LABEL: 0

TEXT: Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes on.<br />

In [ ]:
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

test_reviews = datasets.load_dataset("imdb", split="test")
train_pairs = ReviewPairsDataset(train_reviews, scorer_tokenizer, PREFERRED_LABEL)
test_pairs = ReviewPairsDataset(test_reviews, scorer_tokenizer, PREFERRED_LABEL)

pair_count = len(train_pairs.preferred_reviews)
correct_orderings = 0
for pair_index in range(pair_count):
    preferred_input = scorer_tokenizer(
        train_pairs.preferred_reviews[pair_index], truncation=True, return_tensors="pt"
    ).to(runtime_device)
    other_input = scorer_tokenizer(
        train_pairs.other_reviews[pair_index], truncation=True, return_tensors="pt"
    ).to(runtime_device)
    with torch.no_grad():
        preferred_reward = scorer_model(**preferred_input).logits[0, PREFERRED_LABEL].item()
        other_reward = scorer_model(**other_input).logits[0, PREFERRED_LABEL].item()
    if preferred_reward > other_reward:
        correct_orderings += 1
print("Train result:", correct_orderings / pair_count)

pair_count = len(test_pairs.preferred_reviews)
correct_orderings = 0
for pair_index in range(pair_count):
    preferred_input = scorer_tokenizer(
        test_pairs.preferred_reviews[pair_index], truncation=True, return_tensors="pt"
    ).to(runtime_device)
    other_input = scorer_tokenizer(
        test_pairs.other_reviews[pair_index], truncation=True, return_tensors="pt"
    ).to(runtime_device)
    with torch.no_grad():
        preferred_reward = scorer_model(**preferred_input).logits[0, PREFERRED_LABEL].item()
        other_reward = scorer_model(**other_input).logits[0, PREFERRED_LABEL].item()
    if preferred_reward > other_reward:
        correct_orderings += 1

print("Test result:", correct_orderings / pair_count)

Found 12500 chosen and 12500 rejected texts, 156250000 pairs
Found 12500 chosen and 12500 rejected texts, 156250000 pairs
Train result: 0.98264
Test result: 0.96912


### Reward-guided generation

Before any RL let's see if we can align model without any training. We'll use the technique called __reward-guided inference__: generate $N$ samples, then select the one with the highest reward according to our model.

In [ ]:
sample_inputs = policy_tokenizer(["It was"] * 5, return_tensors="pt").to(runtime_device)
sampled_sequences = policy_model.generate(
    **sample_inputs, do_sample=True, max_new_tokens=50
)
for sequence in sampled_sequences:
    print("Sample:", policy_tokenizer.decode(sequence.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was also filmed on stage with actors from other movies, although some, like "Babylon 5" and "The Patriot", feature characters of different nationalities. All told, it's an interesting and well-written film, and one of films where
Sample: It was hard to believe that the film would be made on DVD by a company that was trying to build up "Star Trek IV: The Voyage Home", which became a successful franchise and is quite successful after so many great movies.<br /><br />
Sample: It was like the second time you watch TV in the world. But you are bored and want a movie.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was a good movie. N

In [ ]:
seed_prompts = ["The movie is", "It was", "The cast is", "The plot", "Plot twist"]

reward_batches = []
generated_batches = []
for seed_text in seed_prompts:
    repeated_prompt = policy_tokenizer(
        [seed_text] * 16, return_tensors="pt"
    ).to(runtime_device)
    generated_tokens = policy_model.generate(
        **repeated_prompt, do_sample=True, max_new_tokens=50
    )
    decoded_candidates = policy_tokenizer.batch_decode(generated_tokens)

    scorer_inputs = scorer_tokenizer(
        decoded_candidates, truncation=True, padding=True, return_tensors="pt"
    ).to(runtime_device)
    with torch.no_grad():
        candidate_rewards = scorer_model(**scorer_inputs).logits[:, PREFERRED_LABEL].cpu().numpy()
    reward_batches.append(candidate_rewards)
    print(decoded_candidates[0])
    generated_batches.append(decoded_candidates)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


The movie is about a scientist, a doctor, a scientist, a doctor who just happens to be a doctor who had just died from a fatal stroke and is making progress. He's going to be able to make some very effective medicine. The doctor, by the


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


It was a shame that the movie was released in the U.S. because it was a very low-key, low-budget, low-budget disaster.<br /><br />The story was very simple: A teenager and his father must fight against


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


The cast is great. But there will be many more...<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


The plot involves two friends from high school going out on a date. When they arrive home about 15 minutes before the date, the girls realize that they must be about to be murdered. They head out for daylight and are attacked by a bunch of crazed teenagers
Plot twist ending is really nothing to be embarrassed about. The story is the most stupid of all the "old-school"-filmed "horror" movies out there. The acting also has to go to show some promise, but to be honest, I


In [ ]:
selected_prompt = "The movie is"
prompt_position = seed_prompts.index(selected_prompt)
best_position = reward_batches[prompt_position].argmax(axis=-1)
generated_batches[prompt_position][best_position]

'The movie is still a little silly, but nothing really different is put to the test from what the characters get out of the movie. My first real comment may be the ending. I hate this movie when the actors end up being "on the safe side of the'

In [ ]:
selected_prompt = "The plot"
prompt_position = seed_prompts.index(selected_prompt)
best_position = reward_batches[prompt_position].argmax(axis=-1)
generated_batches[prompt_position][best_position]

"The plot is as stupid and as predictable as an all out movie. I hate seeing people who hate this movie think the plot is a bunch of random dumb jokes or jokes which are just plain stupid. I think you really can't even call the movie stupid and"

In [ ]:
selected_prompt = "It was"
prompt_position = seed_prompts.index(selected_prompt)
best_position = reward_batches[prompt_position].argmax(axis=-1)
generated_batches[prompt_position][best_position]

"It was good, but just terrible. I didn't enjoy it at all for the first 30 minutes. And then, suddenly my whole childhood became a complete disaster. I felt like a freak out. At first I was completely mad at all this stupid stuff that"

# 2. Fine-tune the main model with reinforcement learning


Here we will align GPT2 to produce negative movie reviews using the reward model we have trained.

The flow is the following: LLM generates its own sentences on each training step. Then, the reward model calculates the reward of those sentences, and using these rewards we update the LLM to increase the probability of sentences with high reward.

The update depends on the specific RL algorithm called [Proximal Policy Optimization(PPO)](https://arxiv.org/abs/1707.06347).

Before we run these stages, we need to create a dataset of "queries" - partial reviews in our case.

In [ ]:
# This code is specific to IMDB
query_dataset = train_reviews.filter(
    lambda review: len(review["text"]) > 200, batched=False
)
query_dataset = query_dataset.remove_columns(["label"])
query_length = trl.core.LengthSampler(2, 8)  # the first 2-8 tokens as query

def attach_tokenized_query(review):
    token_ids = policy_tokenizer.encode(review["text"])[:query_length()]
    review["query"] = policy_tokenizer.decode(token_ids)
    review["input_ids"] = token_ids  # to avoid retokenizing later
    return review

query_dataset = query_dataset.map(attach_tokenized_query, batched=False)
query_dataset.set_format(type="torch")

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/24895 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


Preparing our reward model to predict rewards on generated reviews. We will use plaintext reviews because the LLM uses a different tokenizer from the reward model.

In [ ]:
from typing import List

def score_reviews(review_texts: List[str]) -> torch.Tensor:
    encoded_texts = scorer_tokenizer(
        review_texts, truncation=True, padding=True, return_tensors="pt"
    ).to(runtime_device)
    with torch.no_grad():
        return scorer_model(**encoded_texts).logits[:, 0]

In [ ]:
score_reviews([train_reviews[45]["text"], train_reviews[16000]["text"]])  # test on human-written reviews

tensor([ 4.8359, -4.8789], device='cuda:0')

Moving to RL training, we'll train only LoRA adapters and not the full model.

In [ ]:
import peft
lora_settings = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

policy_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
policy_tokenizer.pad_token = policy_tokenizer.eos_token

policy_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained(
    "lvwerra/gpt2-imdb", device_map=runtime_device
)
policy_model = peft.get_peft_model(
    policy_model, lora_settings, adapter_name="default"
)
policy_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9391


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Using the specific trl trainer that minimize PPO loss

In [ ]:
ppo_config = trl.PPOConfig(
    model_name=policy_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    batch_size=64,
    mini_batch_size=1,
    ppo_epochs=4,
)

ppo_trainer = trl.PPOTrainer(
    ppo_config,
    model=policy_model.model,
    tokenizer=policy_tokenizer,
    dataset=query_dataset,
    data_collator=lambda rows: {key: [row[key] for row in rows] for key in rows[0]},
)

In [ ]:
from tqdm.auto import tqdm
ppo_step_limit = 50
rollout_options = {
    "min_length": -1,
    "max_new_tokens": 128,
    "do_sample": True,
    "top_k": 0,
    "top_p": 1.0,
    "pad_token_id": policy_tokenizer.eos_token_id,
}

with tqdm(enumerate(ppo_trainer.dataloader), total=ppo_step_limit) as progress:
    for step_index, rollout_batch in progress:
        if step_index >= ppo_step_limit:
            break

        # Rollout
        response_tokens = ppo_trainer.generate(
            rollout_batch["input_ids"], **rollout_options
        )

        rollout_batch["response"] = [
            policy_tokenizer.decode(tokens.squeeze()) for tokens in response_tokens
        ]

        # Evaluation
        batch_rewards = score_reviews(rollout_batch["response"])

        # Update
        step_stats = ppo_trainer.step(
            rollout_batch["input_ids"], response_tokens, list(batch_rewards.split(1))
        )
        step_stats["rewards/mean"] = batch_rewards.mean().item()

        print("-" * 30, "STEP", step_index, "-" * 30)
        print(f'rewards/mean:\t{step_stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
        print(f'ppo/returns/mean:\t{step_stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
        print(f'objective/kl:\t{step_stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
        print()

        ppo_trainer.log_stats(
            step_stats, rollout_batch, list(batch_rewards.split(1))
        )

  0%|          | 0/50 [00:00<?, ?it/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


------------------------------ STEP 0 ------------------------------
rewards/mean:	-0.423516273	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.696308494	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)

------------------------------ STEP 1 ------------------------------
rewards/mean:	-0.531005859	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.592206717	<---- model-estimated average discounted reward
objective/kl:	-0.368047923	<---- how far we are from the original model (regularizer)

------------------------------ STEP 2 ------------------------------
rewards/mean:	-0.052177429	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.603630185	<---- model-estimated average discounted reward
objective/kl:	-0.086598746	<---- how far we are from the original model (regularizer)

------------------------------ STEP

In [ ]:
test_prompt = "The movie is "
test_input = policy_tokenizer(test_prompt, return_tensors="pt").to(runtime_device)
generated_output = policy_model.base_model.model.pretrained_model.generate(
    **test_input, do_sample=True, max_new_tokens=50
)

generated_output = policy_tokenizer.decode(generated_output[0])

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [ ]:
generated_output

"The movie is icky, I was just really not satisfied with it. Even the music, the camera was very loud. the characters were awful because they had to have an ending for them, they didn't, but there was no ending in the movie. Maybe there"